In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import importlib.util
import sys
from joblib import Parallel, delayed
import numpy as np

def _find_pipeline_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data_analysis.py").is_file() and (candidate / "three_D_funcs.py").is_file():
            return candidate
    raise RuntimeError("Could not locate the cleaned pipeline root from the notebook location.")

pkg_dir = _find_pipeline_root(Path.cwd())
pipeline_folder = pkg_dir.name
da_file = pkg_dir / "data_analysis.py"

for stale in [
    "data_analysis",
    "three_D_funcs",
    "scwf_anisotropy",
    "scwf_anisotropy.three_D_funcs",
    "scwf_anisotropy.data_analysis",
    pipeline_folder,
    f"{pipeline_folder}.three_D_funcs",
    f"{pipeline_folder}.data_analysis",
    f"{pipeline_folder}_data_analysis",
]:
    sys.modules.pop(stale, None)

da_spec = importlib.util.spec_from_file_location(f"{pipeline_folder}_data_analysis", da_file)
if da_spec is None or da_spec.loader is None:
    raise RuntimeError(f"Could not load pipeline front door from {da_file}")

data_analysis = importlib.util.module_from_spec(da_spec)
da_spec.loader.exec_module(data_analysis)

from functions import general_functions as func

assert hasattr(data_analysis, "run_logscale_filterbank_analysis")

In [2]:
print("pkg_dir =", pkg_dir)
print("da_file =", da_file)

pkg_dir = C:\Users\nokni\work\MHDTurbPy\functions\scwf_cleaned_pkg_rev4
da_file = C:\Users\nokni\work\MHDTurbPy\functions\scwf_cleaned_pkg_rev4\data_analysis.py


In [3]:
sc = "WIND"
lp = rf"C:\Users\nokni\work\WIND_3D\data\3_sec\\"

n_jobs                    = 12
overwrite_existing_files  = False

return_flucs              = True
estimate_alignment_angle  = True
return_B_in_vel_units     = True
use_local_polarity        = True
consider_Vsc              = False

strict_thresh             = False
extra_conditions          = True
only_general              = 1
thetas_phis_step          = 5

theta_thresh_gen = 0 if only_general == 1 else None
phi_thresh_gen = 0 if only_general == 1 else None

conditions = {
    "ell_perp": {"theta": 80, "phi": 80},
    "Ell_perp": {"theta": 80, "phi": 10},
    "ell_par": {"theta": 10, "phi": 90},
    "ell_par_rest": {"theta": 10, "phi": 10},
}
if strict_thresh:
    conditions = {
        "ell_perp": {"theta": 85, "phi": 85},
        "Ell_perp": {"theta": 85, "phi": 5},
        "ell_par": {"theta": 5, "phi": 90},
        "ell_par_rest": {"theta": 5, "phi": 5},
    }

qorder = np.arange(1, 8)
wname            = "mw8"
method_token     = "scwf"
output_subdir    = None
file_name_root   = None
max_interval_dur = 240

ts_list = [
    "dB",
    "dV",
    "dzp",
    "dzm",
    
    "phis",
    "thetas",
    
    "Vsw",
    "Bmod",


    "sig_c_ts",
    "sig_r_ts",

    "sins_zpm_num",
    "sins_zpm_den",

    "parallel_energy_fraction_B",


    "local_polarity",

    'l_mag',
    "l_ell",
    "l_lambda",
    "l_xi",
    "W_B_nT_mag",
]

In [4]:
fnames = func.load_files(lp, "final.pkl")

results = Parallel(n_jobs=n_jobs)(
    delayed(data_analysis.run_logscale_filterbank_analysis)(
        i=i,
        fnames=fnames,
        credentials=None,
        conditions=conditions,
        return_flucs=return_flucs,
        consider_Vsc=consider_Vsc,
        Estimate_5point=False,
        keep_wave_coeefs=False,
        strict_thresh=int(strict_thresh),
        max_hours=300.0,
        qorder=qorder,
        estimate_alignment_angle=estimate_alignment_angle,
        return_mag_align_correl=False,
        only_general=only_general,
        phi_thresh_gen=phi_thresh_gen,
        theta_thresh_gen=theta_thresh_gen,
        sc=sc,
        extra_conditions=extra_conditions,
        ts_list=ts_list,
        overwrite_existing_files=overwrite_existing_files,
        thetas_phis_step=thetas_phis_step,
        return_B_in_vel_units=return_B_in_vel_units,
        max_interval_dur=max_interval_dur,
        use_local_polarity=use_local_polarity,
        wname=wname,
        output_subdir=output_subdir,
        file_name_root=file_name_root,
        method_token=method_token,
    )
    for i in range(len(fnames))
)

C:\Users\nokni\work\WIND_3D\data\3_sec\*\final.pkl
